In [1]:
import sympy as sp

In [4]:
class OptimizationSolver:

    def __init__(self, variables, function, decimal_places=7):
        self.variables = variables
        self.function = function
        self.decimal_places = decimal_places

        # Calculate Gradient
        self.gradient = [
            sp.diff(function, var)
            for var in variables
        ]

        # Calculate Hessian
        self.hessian = sp.hessian(function, variables)

    # Format numerical values
    def format_number(self, value):
        value = sp.N(value, 20)
        return f"{float(value):.{self.decimal_places}f}"

    # Format polynomial expressions
    def format_expression(self, expression):
        try:
            polynomial = sp.Poly(
                expression,
                *self.variables
            )

            terms = []

            for powers, coefficient in polynomial.terms():
                coefficient = float(sp.N(coefficient))

                if abs(coefficient) < 1e-15:
                    continue

                sign = "-" if coefficient < 0 else "+"
                number = self.format_number(abs(coefficient))

                monomial = []

                for var, power in zip(self.variables, powers):
                    if power == 1:
                        monomial.append(str(var))
                    elif power > 1:
                        monomial.append(f"{var}**{power}")

                term = number

                if monomial:
                    term += "*" + "*".join(monomial)

                terms.append((sign, term))

            if not terms:
                return self.format_number(0)

            result = ""

            for index, (sign, term) in enumerate(terms):
                if index == 0:
                    result += ("-" if sign == "-" else "") + term
                else:
                    result += f" {sign} {term}"

            return result

        except sp.PolynomialError:
            return str(sp.N(expression, self.decimal_places + 5))

    # 1. Display Function
    def show_function(self):
        print("\n===== FUNCTION =====")
        print(self.format_expression(self.function))

    # 2. Display Gradient
    def show_gradient(self):
        print("\n===== GRADIENT =====")

        for var, derivative in zip(
            self.variables, self.gradient
        ):
            print(
                f"df/d{var} = "
                f"{self.format_expression(derivative)}"
            )

    # 3. Find ALL stationary points
    # def find_stationary_points(self):
    #     print("\n===== STATIONARY POINTS =====")

    #     equations = [
    #         sp.Eq(derivative, 0)
    #         for derivative in self.gradient
    #     ]

    #     solutions = sp.solve(
    #         equations,
    #         self.variables,
    #         dict=True
    #     )

    #     # Keep only complete solutions
    #     solutions = [
    #         solution
    #         for solution in solutions
    #         if all(var in solution for var in self.variables)
    #     ]

    #     if not solutions:
    #         print("No stationary points found.")
    #         return []

    #     print(f"Total stationary points: {len(solutions)}")

    #     for index, solution in enumerate(solutions, 1):
    #         print(f"\nPoint {index}:")

    #         for var in self.variables:
    #             print(
    #                 f"  {var} = "
    #                 f"{self.format_number(solution[var])}"
    #             )

    #     return solutions

    # 3. Find ALL stationary points
    def find_stationary_points(self):
        print("\n===== STATIONARY POINTS =====")

        equations = [
            sp.Eq(derivative, 0)
            for derivative in self.gradient
        ]

        solutions = sp.solve(
            equations,
            self.variables,
            dict=True
        )

        # Keep only complete solutions
        solutions = [
            solution
            for solution in solutions
            if all(var in solution for var in self.variables)
        ]

        if not solutions:
            print("No stationary points found.")
            return []

        print(f"Total stationary points: {len(solutions)}")

        for index, solution in enumerate(solutions, 1):
            print(f"\nPoint {index}:")

            for var in self.variables:
                value = sp.simplify(solution[var])

                print(f"  {var} = {value}")

        return solutions

    # 4. Display symbolic Hessian
    def show_hessian(self):
        print("\n===== HESSIAN MATRIX =====")
        sp.pprint(self.hessian)

    # 5. Calculate Hessian at each stationary point
    def calculate_minors(self, point):
        hessian_at_point = self.hessian.subs(point)

        print("\nHessian at this point:")
        for i in range(len(self.variables)):
            print(
                [
                    self.format_number(hessian_at_point[i, j])
                    for j in range(len(self.variables))
                ]
            )

        print("\n===== PRINCIPAL MINORS =====")

        minors = []

        for k in range(1, len(self.variables) + 1):

            sub_matrix = hessian_at_point[:k, :k]

            determinant = sp.simplify(sub_matrix.det())

            minors.append(determinant)

            print(
                f"D{k} = "
                f"{self.format_number(determinant)}"
            )

        return minors, hessian_at_point

    # 6. Classify stationary point
    def classify_point(self, minors, hessian_at_point):

        values = [
            float(sp.N(d, 20))
            for d in minors
        ]

        tolerance = 1e-9

        # Sylvester criterion: Positive definite
        positive_definite = all(
            value > tolerance
            for value in values
        )

        if positive_definite:
            return "LOCAL MINIMUM"

        # Sylvester criterion: Negative definite
        negative_definite = all(
            ((-1) ** (index + 1)) * value > tolerance
            for index, value in enumerate(values)
        )

        if negative_definite:
            return "LOCAL MAXIMUM"

        # Check eigenvalues to identify saddle points
        eigenvalues = hessian_at_point.evalf(20).eigenvals()

        eigenvalues = [
            float(sp.re(sp.N(value, 20)))
            for value in eigenvalues.keys()
        ]

        has_positive = any(
            value > tolerance for value in eigenvalues
        )

        has_negative = any(
            value < -tolerance for value in eigenvalues
        )

        has_zero = any(
            abs(value) <= tolerance for value in eigenvalues
        )

        if has_positive and has_negative:
            return "SADDLE POINT"

        if has_zero:
            return "INCONCLUSIVE (Degenerate Hessian)"

        if has_positive:
            return "LOCAL MINIMUM (Semidefinite Hessian)"

        if has_negative:
            return "LOCAL MAXIMUM (Semidefinite Hessian)"

        return "INCONCLUSIVE"

    # 7. Solve optimization problem
    def solve(self):

        print("\n================================")
        self.show_function()
        self.show_gradient()
        self.show_hessian()

        stationary_points = self.find_stationary_points()

        if not stationary_points:
            return

        print("\n================================")
        print("===== OPTIMIZATION RESULTS =====")

        for index, point in enumerate(stationary_points, 1):

            print(f"\n\n######## POINT {index} ########")

            print("\nStationary point:")

            # for var in self.variables:
            #     print(
            #         f"{var} = "
            #         f"{self.format_number(point[var])}"
            #     )

            for var in self.variables:
                value = sp.simplify(point[var])
                print(f"{var} = {value}")

            # Calculate minors for CURRENT point
            minors, hessian_at_point = (
                self.calculate_minors(point)
            )

            # Classify CURRENT point
            result = self.classify_point(
                minors,
                hessian_at_point
            )

            print("\nClassification:", result)

            # Calculate objective function
            function_value = self.function.subs(point)

            print(
                "f(x*) =",
                self.format_number(function_value)
            )


In [38]:
# ==================================================
# EXAMPLE 4.1.9 - TWO VARIABLES
# ==================================================

x1, x2 = sp.symbols("x1 x2")

f1 = (
    -sp.Rational(7, 2)
    + sp.Rational(123, 10) * x1
    + x1**2 / 2
    - 41 * x2
    - 2 * x1 * x2
    + 4 * x2**2
)

print("\n\n######## EXAMPLE 4.1.9 ########")

solver1 = OptimizationSolver(
    variables=[x1, x2],
    function=f1
)

solver1.solve()




######## EXAMPLE 4.1.9 ########

FUNCTION:
  2                                       
x₁              123⋅x₁       2           7
─── - 2⋅x₁⋅x₂ + ────── + 4⋅x₂  - 41⋅x₂ - ─
 2                10                     2

===== GRADIENT =====
df/dx1 = x1 - 2*x2 + 123/10
df/dx2 = -2*x1 + 8*x2 - 41

===== HESSIAN MATRIX =====
⎡1   -2⎤
⎢      ⎥
⎣-2  8 ⎦

===== STATIONARY POINTS =====
Point 1: {x1: -41/10, x2: 41/10}

===== PRINCIPAL MINORS =====
D1 = 1
D2 = 4

===== OPTIMIZATION RESULT =====

Stationary point:
x1 = -41/10
x2 = 41/10
Classification: LOCAL MINIMUM
f(x*) = -112.7650000


In [5]:
# ==================================================
# EXAMPLE 4.10 - THREE VARIABLES
# ==================================================

# x, y, z = sp.symbols("x y z")

# f2 = (
#     + x**3 + y**3 - z**2 + x*y + 2*x*z
# )

# print("\n\n######## EXAMPLE 4.10 ########")

# solver2 = OptimizationSolver(
#     variables=[x, y, z],
#     function=f2
# )
# solver2.solve()

x, y, z = sp.symbols("x y z")

f2 = (
    x**3
    + x*y
    + 2*x*z
    + y**3
    - z**2
)

solver = OptimizationSolver(
    variables=[x, y, z],
    function=f2,
    # decimal_places=7
)

solver.solve()




===== FUNCTION =====
1.0000000*x**3 + 1.0000000*x*y + 2.0000000*x*z + 1.0000000*y**3 - 1.0000000*z**2

===== GRADIENT =====
df/dx = 3.0000000*x**2 + 1.0000000*y + 2.0000000*z
df/dy = 1.0000000*x + 3.0000000*y**2
df/dz = 2.0000000*x - 2.0000000*z

===== HESSIAN MATRIX =====
⎡6⋅x   1   2 ⎤
⎢            ⎥
⎢ 1   6⋅y  0 ⎥
⎢            ⎥
⎣ 2    0   -2⎦

===== STATIONARY POINTS =====
Total stationary points: 4

Point 1:
  x = -1/3
  y = 1/3
  z = -1/3

Point 2:
  x = 0
  y = 0
  z = 0

Point 3:
  x = -1/2 - sqrt(5)/6
  y = -sqrt(5)/6 - 1/6
  z = -1/2 - sqrt(5)/6

Point 4:
  x = -1/2 + sqrt(5)/6
  y = -1/6 + sqrt(5)/6
  z = -1/2 + sqrt(5)/6

===== OPTIMIZATION RESULTS =====


######## POINT 1 ########

Stationary point:
x = -1/3
y = 1/3
z = -1/3

Hessian at this point:
['-2.0000000', '1.0000000', '2.0000000']
['1.0000000', '2.0000000', '0.0000000']
['2.0000000', '0.0000000', '-2.0000000']

===== PRINCIPAL MINORS =====
D1 = -2.0000000
D2 = -5.0000000
D3 = 2.0000000

Classification: SADDLE POI